In [1]:
import pandas as pd
import numpy as np
import pickle
import pandas_datareader.data as web
from datetime import datetime

In [ ]:
overwrite_risk_free_rates = False

In [3]:
rf = web.DataReader(
    'DGS3MO',
    'fred',
    start='2007-02-28',
    end='2025-07-30'
)

print(rf.min())
print(rf.max())

DGS3MO    0.0
dtype: float64
DGS3MO    5.63
dtype: float64


In [4]:
rf = rf.sort_index().replace([np.inf, -np.inf], np.nan)
full_daily_index = pd.date_range(rf.index.min(), rf.index.max(), freq='D')
rf = rf.reindex(full_daily_index)
rf['DGS3MO'] = rf['DGS3MO'].ffill()

print(rf)

            DGS3MO
2007-02-28    5.16
2007-03-01    5.15
2007-03-02    5.12
2007-03-03    5.12
2007-03-04    5.12
...            ...
2025-07-26    4.42
2025-07-27    4.42
2025-07-28    4.40
2025-07-29    4.40
2025-07-30    4.41

[6728 rows x 1 columns]


In [5]:
rf = rf[rf.index.weekday == 2]

print(rf)

            DGS3MO
2007-02-28    5.16
2007-03-07    5.12
2007-03-14    5.06
2007-03-21    5.05
2007-03-28    5.06
...            ...
2025-07-02    4.41
2025-07-09    4.42
2025-07-16    4.41
2025-07-23    4.41
2025-07-30    4.41

[962 rows x 1 columns]


In [6]:
rf['DGS3MO'] = (1 + rf['DGS3MO'] / 100)**(1/52) - 1

print(rf)

              DGS3MO
2007-02-28  0.000968
2007-03-07  0.000961
2007-03-14  0.000950
2007-03-21  0.000948
2007-03-28  0.000950
...              ...
2025-07-02  0.000830
2025-07-09  0.000832
2025-07-16  0.000830
2025-07-23  0.000830
2025-07-30  0.000830

[962 rows x 1 columns]


In [7]:
rf['DGS3MO'] = rf['DGS3MO'].shift(1)
rf = rf.iloc[1:]

print(rf)

              DGS3MO
2007-03-07  0.000968
2007-03-14  0.000961
2007-03-21  0.000950
2007-03-28  0.000948
2007-04-04  0.000950
...              ...
2025-07-02  0.000825
2025-07-09  0.000830
2025-07-16  0.000832
2025-07-23  0.000830
2025-07-30  0.000830

[961 rows x 1 columns]


In [8]:
rf_list = rf['DGS3MO'].tolist()

print(rf_list)

[0.0009680223444352709, 0.0009606990365178536, 0.000949708948558392, 0.0009478766687753826, 0.000949708948558392, 0.0009515410573002203, 0.0009460442179187734, 0.0009387127031117437, 0.0009332122691525502, 0.000922206774399692, 0.0009167017118587584, 0.0008928285807847658, 0.000922206774399692, 0.0009111951047087175, 0.0009020139912823133, 0.0008762840002949357, 0.0008909909826793072, 0.0008965032608923007, 0.0009295444565740052, 0.0009295444565740052, 0.0009313784485454057, 0.0009368793964033006, 0.0009185369042856717, 0.0009295444565740052, 0.000793351229358974, 0.0006933670245310442, 0.0007508276332766073, 0.0008210344467252106, 0.0007600797353661815, 0.0007415711667333458, 0.000704501572734495, 0.0007471255706263946, 0.0007656289034105868, 0.000756379417998776, 0.000726751731815023, 0.0007434228093978756, 0.0006599254183341507, 0.0006450441671219664, 0.0005854060355812152, 0.0005779385065320941, 0.000581672626345453, 0.0005461697599027371, 0.0005517797486993548, 0.00062828925720759

In [9]:
start_str = rf.index[0].strftime('%Y-%m-%d')
end_str   = rf.index[-1].strftime('%Y-%m-%d')

fname = f"risk_free_returns_weekly_{start_str}_to_{end_str}_inclusive.pkl"

print(fname)

risk_free_returns_weekly_2007-03-07_to_2025-07-30_inclusive.pkl


In [10]:
if overwrite_risk_free_rates:

    with open(f"/dcs/pg24/u5674159/mis-dro-code/james-data/{fname}", "wb") as f:
        pickle.dump(rf_list, f)